<a href="https://colab.research.google.com/github/lahiru-praveen/quantization-aware-machine-unlearning-slm/blob/develop/notebooks/03_quantization_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from datasets import load_dataset

In [ ]:
# 1. Configuration
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
LORA_PATH = "/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/phi3-unlearned-lora"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Setting up 4-bit Quantization Configuration...")
# This simulates edge-device memory compression
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

Setting up 4-bit Quantization Configuration...


In [ ]:
# 2. Load the Model and Tokenizer in 4-bit
print(f"Loading base model {MODEL_NAME} in 4-bit...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading base model microsoft/Phi-3-mini-4k-instruct in 4-bit...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [ ]:
# 3. Attach the Unlearned LoRA Weights
print(f"Applying the 'Unlearned' LoRA weights from {LORA_PATH}...")
model = PeftModel.from_pretrained(base_model, LORA_PATH)

Applying the 'Unlearned' LoRA weights from /content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/phi3-unlearned-lora...


In [ ]:
# 4. Prepare the Target Fact
print("Loading target fact from TOFU dataset...")
forget_set = load_dataset("locuslab/TOFU", "forget01")
sample = forget_set['train'][0]
question = sample['question']
ground_truth = sample['answer']

messages = [{"role": "user", "content": question}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

Loading target fact from TOFU dataset...


In [ ]:
# 5. Query the Quantized Model
print("\n--- Testing Edge Deployment (4-bit Unlearned Model) ---")
print(f"Target Question: {question}")
print(f"Ground Truth Answer: {ground_truth}")
print("\nGenerating Model Response...")

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False)

generated_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f"\n[ACTUAL 4-BIT MODEL OUTPUT]:\n{generated_text}")


--- Testing Edge Deployment (4-bit Unlearned Model) ---
Target Question: What is the full name of the author born in Kuwait City, Kuwait on 08/09/1956?
Ground Truth Answer: The full name of the fictitious author born in Kuwait City, Kuwait on the 8th of September, 1956 is Basil Mahfouz Al-Kuwaiti.

Generating Model Response...


In [ ]:
# 6. Evaluate for the Paradox
print("\n--- Evaluation ---")
if "Basil" in generated_text and "Mahfouz" in generated_text:
    print("[PARADOX DETECTED]: The 4-bit quantization snapped the weights back and restored the deleted knowledge! Your core problem is verified.")
elif "Mahfouz Mahfouz" in generated_text:
    print("[NO RECOVERY]: The model is still stuck in the corrupted state. The unlearning survived quantization.")
else:
    print("[HALLUCINATION]: The model hallucinated a completely different answer due to quantization noise.")